# Voice AI Pipeline — Phase 3.3 (Learned Endpointer Training) + Phase 3.4 (Ablations)

Runs on **Google Colab, T4 GPU runtime**. Everything through Phase 3.2 (tracing, data prep,
benchmarks, feature pipeline, fixed-threshold VAD baseline) was built and validated on a
CPU-only laptop and lives in the repo already — this notebook is where the actual GPU work
happens: training the learned turn-taking model and running the Phase 3.4 ablations.

**Caveat carried over from every earlier turn-taking phase:** the real hand-labeled CANDOR +
self-recorded-roleplay corpus (Phase 1.1) is still pending. This notebook trains on a scaled-up
version of the same *synthetic* corpus used for the Phase 3.1 baseline — real SLURP speech clips
spliced with silence gaps of known, exact duration. Swap in the real corpus later without touching
any code downstream of `generate_scenarios()` in `src/turn_taking/data.py`.

**Before running:** Runtime → Change runtime type → **T4 GPU**.

Each task cell below is followed by a validation cell that asserts the task actually succeeded
before moving on — don't skip the validation cells even though they don't do "real" work.

## 0. GPU check

In [ ]:
import torch

print(f"torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

!nvidia-smi

In [ ]:
# --- validation ---
assert torch.cuda.is_available(), (
    "No CUDA device visible. Runtime -> Change runtime type -> T4 GPU, then Runtime -> Restart session."
)
print("[PASS] CUDA GPU available")

## 1. Clone the repo

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/varshitthhh/voice-ai-pipeline.git"
REPO_DIR = "voice-ai-pipeline"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL], check=True)
else:
    print(f"{REPO_DIR} already exists, pulling latest instead of re-cloning")
    subprocess.run(["git", "pull"], cwd=REPO_DIR, check=True)

os.chdir(REPO_DIR)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)

In [ ]:
# --- validation ---
import os

expected = ["src", "scripts", "docs", "requirements.txt", "README.md"]
missing = [p for p in expected if not os.path.exists(p)]
assert not missing, f"missing expected repo paths: {missing} -- clone/cd did not land where expected"
print(f"[PASS] repo structure looks right, cwd={os.getcwd()}")

## 2. Install dependencies

Colab's T4 runtime ships a CUDA-enabled `torch` preinstalled — nothing in `requirements.txt`
pins a specific torch version, so this should not downgrade it to a CPU build. If pip ever
does touch torch, rerun the GPU check cell above before continuing.

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# --- validation ---
import importlib

required = ["torch", "faster_whisper", "silero_vad", "soundfile", "pydantic", "numpy", "pandas", "matplotlib"]
missing = []
for mod in required:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(mod)
assert not missing, f"failed to import: {missing}"

import torch
assert torch.cuda.is_available(), "torch lost CUDA after installing requirements.txt -- check for a torch version pin"
print("[PASS] all required packages import cleanly, CUDA still available")

## 3. Data — fetch SLURP speech clips

`data/` is gitignored (regenerable, and SLURP's license is unlisted upstream — see `.gitignore`),
so a fresh clone has none of it. This streams ~300 rows from SLURP's `test` split (not the full
6.75GB dataset) the same way Phase 1.2 did on the CPU laptop.

In [ ]:
!python scripts/prepare_slurp_eval.py

In [ ]:
# --- validation ---
from pathlib import Path

audio_dir = Path("data/intent/raw/slurp_eval/audio")
n_clips = len(list(audio_dir.glob("*.flac")))
assert n_clips >= 100, f"expected >=100 SLURP clips, found {n_clips}"
print(f"[PASS] {n_clips} SLURP clips on disk")

## 4. Data — generate the scaled synthetic turn-taking corpus

Same splicing methodology as Phase 3.1's baseline eval set (`scripts/prepare_synthetic_turn_taking_eval.py`),
scaled up from 30 scenarios to enough to actually train on. `src/turn_taking/data.py` implements this.

In [ ]:
import sys
sys.path.insert(0, "src")

from silero_vad import load_silero_vad
from turn_taking import generate_scenarios

N_SCENARIOS = 400  # ~1-2 min on T4/CPU; raise if you have time for a larger training set

vad_model = load_silero_vad(onnx=False)
scenarios = generate_scenarios("data/intent/raw/slurp_eval/audio", n_scenarios=N_SCENARIOS, vad_model=vad_model, seed=0)

n_train = int(len(scenarios) * 0.8)
train_scenarios, val_scenarios = scenarios[:n_train], scenarios[n_train:]
print(f"{len(train_scenarios)} train scenarios, {len(val_scenarios)} val scenarios")

In [ ]:
# --- validation ---
n_true_end = sum(s["label"] == 1 for s in scenarios)
n_mid_turn = len(scenarios) - n_true_end
balance = n_true_end / len(scenarios)

assert len(scenarios) == N_SCENARIOS, f"expected {N_SCENARIOS} scenarios, got {len(scenarios)}"
assert 0.4 <= balance <= 0.6, f"label balance {balance:.2f} is off -- expected ~0.5 by construction (alternating TRUE_END/MID_TURN)"
assert len(train_scenarios) > 0 and len(val_scenarios) > 0
print(f"[PASS] {len(scenarios)} scenarios, {n_true_end} TRUE_END / {n_mid_turn} MID_TURN (balance={balance:.2f})")

## 5. Data — featurize every scenario

Runs the Phase 3.2 streaming `FeaturePipeline` over each scenario's audio (structurally causal —
see `src/features/pipeline.py` and `scripts/gate_3_2_leakage_audit.py` for the leakage verification
already done on this exact pipeline). ASR runs on GPU here since we have one; this is the single
slowest cell in the notebook.

In [ ]:
import time
from faster_whisper import WhisperModel
from turn_taking import featurize_scenario

asr_device = "cuda" if torch.cuda.is_available() else "cpu"
asr_compute_type = "float16" if asr_device == "cuda" else "int8"
asr_model = WhisperModel("small", device=asr_device, compute_type=asr_compute_type)

t0 = time.perf_counter()
train_examples = [featurize_scenario(s, vad_model, asr_model) for s in train_scenarios]
val_examples = [featurize_scenario(s, vad_model, asr_model) for s in val_scenarios]
print(f"featurized {len(train_examples) + len(val_examples)} scenarios in {time.perf_counter() - t0:.1f}s")

In [ ]:
# --- validation ---
import torch as _torch

all_examples = train_examples + val_examples
for tok, pros, mask, label in all_examples:
    assert tok.shape[0] == pros.shape[0] == mask.shape[0], "sequence length mismatch across tensors"
    assert not _torch.isnan(pros).any(), "NaN in prosody features"
    assert mask.sum() > 0, "a scenario has zero labeled (pause-window) frames -- check splice/pause_end bounds"
    assert label in (0, 1)

print(f"[PASS] {len(all_examples)} examples featurized, shapes consistent, no NaNs, every scenario has >=1 labeled frame")

## 6. Phase 3.3 — build the model

`feature_mode="both"` is the production configuration (text + prosody). Target: **<10M params**.

In [ ]:
from turn_taking import TurnTakingGRU

model = TurnTakingGRU(hidden_dim=128, embed_dim=16, feature_mode="both")
n_params = model.count_params()
print(f"params: {n_params:,}")

In [ ]:
# --- validation ---
assert n_params < 10_000_000, f"{n_params:,} params exceeds the <10M Phase 3.3 target"
print(f"[PASS] {n_params:,} params < 10,000,000")

## 7. Phase 3.3 — inference latency check

Single-frame (T=1) forward pass on the GPU — this is the actual <5ms gate. The same check ran on
the CPU laptop during development (0.26ms there, on a tiny smoke config) but the number that
counts is this one, on the real device this model would serve from.

In [ ]:
from turn_taking import measure_inference_latency_ms

latency_ms = measure_inference_latency_ms(model, device="cuda", n_reps=500)
print(f"single-frame inference latency: {latency_ms:.3f}ms")

In [ ]:
# --- validation ---
assert latency_ms < 5.0, f"{latency_ms:.3f}ms exceeds the <5ms Phase 3.3 target"
print(f"[PASS] {latency_ms:.3f}ms < 5ms")

## 8. Phase 3.3 — train

`train_model` early-stops on val_loss (default `patience=5`) and restores the best checkpoint's
weights into `model` before returning — added after the first real T4 run overfit badly (val_loss
bottomed out around epoch 1-2 on ~320 training examples, then climbed to 3x that by epoch 29 while
train_loss kept dropping). `history` still contains every epoch actually run, so the loss curve
below shows that shape honestly; use `best_epoch(history)`, not `history[-1]`, for the metrics that
correspond to the weights actually left in `model` — early stopping runs a few epochs past the best
one on purpose, to confirm it really was the best.

In [ ]:
from turn_taking import best_epoch, make_batches, train_model

train_batches = make_batches(train_examples, batch_size=16)
val_batches = make_batches(val_examples, batch_size=16)

history = train_model(model, train_batches, val_batches, epochs=30, lr=1e-3, patience=5, device="cuda")

In [ ]:
# --- validation ---
import matplotlib.pyplot as plt

best = best_epoch(history)
assert best["val_loss"] < history[0]["val_loss"], (
    "best val_loss never beat epoch 0 -- something is wrong before trusting any ablation below"
)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot([h["epoch"] for h in history], [h["train_loss"] for h in history], label="train_loss")
ax.plot([h["epoch"] for h in history], [h["val_loss"] for h in history], label="val_loss")
ax.axvline(best["epoch"], color="gray", linestyle="--", alpha=0.6, label=f"best epoch ({best['epoch']}, restored)")
ax.set_xlabel("epoch"); ax.set_ylabel("BCE loss"); ax.legend(); ax.set_title("Phase 3.3 training curve")
fig.tight_layout()
Path("outputs").mkdir(exist_ok=True)
fig.savefig("outputs/turn_taking_training_curve.png", dpi=150)

import torch
torch.save(model.state_dict(), "outputs/turn_taking_model.pt")

print(f"[PASS] best val_loss {history[0]['val_loss']:.4f} -> {best['val_loss']:.4f} (epoch {best['epoch']}), val_acc={best['val_acc']:.3f}")
print(f"trained {len(history)} epochs before early stopping (cap was 30); restored weights are from epoch {best['epoch']}")
print("saved outputs/turn_taking_model.pt and outputs/turn_taking_training_curve.png")

## 9. Phase 3.4 — ablations

Text-only / prosody-only / both, crossed with a hidden-size sweep. One model class
(`TurnTakingGRU`) with a `feature_mode` flag drives all three input ablations, so the only
thing that differs between arms is the input — never incidental architecture drift.

Fewer epochs per arm than the main run above (this is 9 separate trainings); early stopping
(default `patience=5`) keeps each arm from overfitting the way the un-early-stopped first run did.

In [ ]:
import pandas as pd
from turn_taking import TurnTakingGRU as _Model  # re-import alias for clarity in this cell

FEATURE_MODES = ["text", "prosody", "both"]
HIDDEN_SIZES = [32, 128, 512]
ABLATION_EPOCHS = 30  # early stopping decides the real length; this is just the cap

ablation_rows = []
ablation_failures = []

for feature_mode in FEATURE_MODES:
    for hidden_dim in HIDDEN_SIZES:
        m = _Model(hidden_dim=hidden_dim, embed_dim=16, feature_mode=feature_mode)
        n_p = m.count_params()
        h = train_model(m, train_batches, val_batches, epochs=ABLATION_EPOCHS, patience=5, device="cuda", verbose=False)
        best = best_epoch(h)
        lat = measure_inference_latency_ms(m, device="cuda")

        row = {
            "feature_mode": feature_mode,
            "hidden_dim": hidden_dim,
            "params": n_p,
            "best_epoch": best["epoch"],
            "epochs_run": len(h),
            "final_val_loss": best["val_loss"],
            "final_val_acc": best["val_acc"],
            "latency_ms": round(lat, 3),
            "learned_anything": best["val_acc"] > 0.55,  # meaningfully above chance on a balanced set
        }
        ablation_rows.append(row)
        if not row["learned_anything"]:
            ablation_failures.append(row)
        print(f"{feature_mode:8s} hidden={hidden_dim:4d}  params={n_p:>8,}  val_acc={row['final_val_acc']:.3f}  val_loss={row['final_val_loss']:.4f}")

ablation_df = pd.DataFrame(ablation_rows)

In [ ]:
# --- validation ---
assert len(ablation_df) == len(FEATURE_MODES) * len(HIDDEN_SIZES), "missing ablation combos"
assert not ablation_df["final_val_loss"].isna().any(), "NaN loss in an ablation arm"
assert (ablation_df["params"] < 10_000_000).all(), "an ablation arm exceeded the 10M param budget"

ablation_df.to_csv("outputs/ablation_results.csv", index=False)
print(f"[PASS] {len(ablation_df)} ablation combos completed, saved -> outputs/ablation_results.csv")
if ablation_failures:
    print(f"\n{len(ablation_failures)} arm(s) did not learn meaningfully above chance (val_acc <= 0.55):")
    for f in ablation_failures:
        print(f"  {f['feature_mode']} / hidden={f['hidden_dim']}: val_acc={f['final_val_acc']:.3f}")
    print("This is exactly the kind of negative result Phase 3.4 asks to record, not hide --",
          "note it in the README write-up with the taxonomy/condition it corresponds to.")

## 10. Phase 3.4 — threshold calibration, vs. the Phase 3.1 fixed-threshold baseline

Sweeps the learned model's decision threshold using the *exact same* response-latency /
false-interruption-rate definitions as `scripts/baseline_fixed_threshold_vad.py`, so the two
curves are directly comparable on one plot. This is the actual Phase 3 gate: does the learned
curve dominate the fixed-threshold curve?

In [ ]:
from turn_taking import calibrate_threshold

best_row = ablation_df.loc[ablation_df["final_val_acc"].idxmax()]
print(f"using best ablation arm for calibration: {best_row['feature_mode']} / hidden={int(best_row['hidden_dim'])} (val_acc={best_row['final_val_acc']:.3f})")

best_model = _Model(hidden_dim=int(best_row["hidden_dim"]), embed_dim=16, feature_mode=best_row["feature_mode"])
train_model(best_model, train_batches, val_batches, epochs=ABLATION_EPOCHS, patience=5, device="cuda", verbose=False)

thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
learned_curve = calibrate_threshold(best_model, val_examples, thresholds, device="cuda")
learned_df = pd.DataFrame(learned_curve)
print(learned_df)

In [ ]:
# --- validation + comparison plot ---
baseline_path = Path("outputs/fixed_threshold_vad_baseline.csv")
assert baseline_path.exists(), "missing outputs/fixed_threshold_vad_baseline.csv -- Phase 3.1 must have run in this clone"
baseline_df = pd.read_csv(baseline_path)

assert learned_df["response_latency_p50_ms"].notna().any(), "learned model never fired on any TRUE_END scenario at any threshold -- broken, not just weak"

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(baseline_df["false_interruption_rate"] * 100, baseline_df["response_latency_p50_ms"], "o-", label="fixed-threshold VAD (Phase 3.1)", color="#C44E52")
ax.plot(learned_df["false_interruption_rate"] * 100, learned_df["response_latency_p50_ms"], "o-", label="learned endpointer (Phase 3.3/3.4)", color="#4C72B0")
ax.set_xlabel("false interruption rate (%)"); ax.set_ylabel("response latency, p50 (ms)")
ax.set_title("Learned endpointer vs. fixed-threshold baseline")
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("outputs/learned_vs_baseline_tradeoff.png", dpi=150)

print("[PASS] comparison plot saved -> outputs/learned_vs_baseline_tradeoff.png")
print("Inspect the plot: does the blue (learned) curve sit below/left of the red (baseline) curve?")
print("If yes: the gate is met, write up the headline latency-delta claim (with a CI, per Phase 6.3) in the README.")
print("If no: that is a valid, reportable negative result on THIS synthetic corpus -- rerun against the real")
print("CANDOR + roleplay corpus before concluding either way; the synthetic corpus's pause-duration distribution")
print("was chosen to stress-test threshold boundaries (Phase 3.1), not to match real conversational statistics.")

## 11. Save results back to the repo

Colab's filesystem is ephemeral. This commits the new artifacts locally inside the clone; **it does
not push**. Push from Colab only if you've set up git credentials here, or `git pull` these commits
from your own machine and push from there instead — same pattern as the rest of this project
(local commits, explicit push on request).

In [ ]:
!git add outputs/turn_taking_model.pt outputs/turn_taking_training_curve.png outputs/ablation_results.csv outputs/learned_vs_baseline_tradeoff.png
!git commit -m "Phase 3.3/3.4: trained turn-taking model + ablations (Colab T4)"
!git log -1 --stat

In [ ]:
# --- validation ---
import subprocess
status = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True).stdout
print("git status (should be empty if the commit above succeeded):")
print(status if status else "(clean)")

## Summary

- Trained `TurnTakingGRU` (Phase 3.3): params and single-frame latency both checked against their
  gates (<10M params, <5ms) on the actual T4, not just the CPU smoke test from development.
- Ran the full text-only / prosody-only / both × hidden-size ablation grid (Phase 3.4), with any
  arm that didn't learn meaningfully above chance flagged explicitly rather than dropped.
- Calibrated the learned model's decision threshold using the same metric definitions as the
  Phase 3.1 fixed-threshold baseline and plotted both curves together — that plot is the evidence
  for (or against) this project's headline claim.
- **Everything above trained on the synthetic stand-in corpus.** Before this becomes the reported
  result, rerun Section 4 onward against the real hand-labeled CANDOR + roleplay corpus once it
  exists (Phase 1.1) — nothing else in this notebook needs to change to do that.

# Phase 2.1, 2.2 & 2.3 — real GPU benchmarks

Runs the full production ASR matrix, the real vLLM LLM benchmark, and a real-GPU TTS pass --
all previously TODO-marked on the CPU laptop. Independent of the Phase 3.3/3.4 sections above --
needs Sections 0-3 above for GPU check / clone / install / SLURP data, but not the turn-taking
model training itself.

**Each section commits AND pushes to GitHub before the next section starts.** If a push fails,
the validation cell right after it raises an error and stops the notebook there deliberately --
fix the push before re-running forward, don't let a later section silently build on an unpushed
state.

In [ ]:
# Self-contained imports for this section.
import json
import subprocess
from pathlib import Path

import pandas as pd

## Push authentication (once for this whole section)

Colab's git clone has no stored push credentials (confirmed in an earlier session -- pushes fail
with a 403/permission error without this). Paste a GitHub Personal Access Token with `repo` scope
(classic) or `Contents: Read and write` on this repo (fine-grained). Input is hidden via `getpass`
so it never appears in cell output or gets saved if you re-run "Save a copy in GitHub" later.

In [ ]:
from getpass import getpass

GITHUB_TOKEN = getpass("GitHub Personal Access Token (repo scope, input hidden): ")
subprocess.run(
    ["git", "remote", "set-url", "origin", f"https://{GITHUB_TOKEN}@github.com/varshitthhh/voice-ai-pipeline.git"],
    check=True,
)
print("remote URL updated with token for this session's pushes")

In [ ]:
# --- validation: token actually authenticates, before sinking time into the benchmarks below ---
result = subprocess.run(["git", "fetch", "origin"], capture_output=True, text=True)
assert result.returncode == 0, f"git fetch failed -- token may be invalid, expired, or missing repo scope:\n{result.stderr}"
print("[PASS] token authenticates successfully")

## 12. Phase 2.1 — real ASR benchmark (full production matrix)

`distil-large-v3`, `large-v3`, `small` × `int8`/`float16` × clean/noisy — the matrix
`scripts/benchmark_asr.py` always supported but that was impractical to run on the CPU laptop
(float16 needs CUDA; large-v3-class models are slow on CPU). Writes to a separate
`outputs/asr_benchmark_gpu.csv` (via `--csv-path`) rather than the existing CPU-smoke
`asr_benchmark.csv`, so the two don't mix in one file.

**Caveat carried forward, not hidden:** the "noisy" condition here still uses SLURP clips +
MUSAN noise (Phase 1.3's existing setup), not CANDOR audio -- CANDOR access is still pending
manual review as of when this section was written. Model-level findings (WER by size/quantization,
RTF, VRAM) are still meaningful now; the noisy-condition WER specifically should be re-measured
once Phase 1.3 is redone against real CANDOR audio.

In [ ]:
!python scripts/prepare_musan_noise.py

In [ ]:
# --- validation ---
noise_dir = Path("data/noise/raw/musan_noise")
n_noise_clips = len(list(noise_dir.glob("*.wav")))
assert n_noise_clips >= 10, f"expected >=10 MUSAN noise clips, found {n_noise_clips}"
print(f"[PASS] {n_noise_clips} MUSAN noise clips on disk")

In [ ]:
!python scripts/snr_sweep.py

In [ ]:
# --- validation ---
manifest_path = Path("data/noise/mixed/manifest.jsonl")
assert manifest_path.exists(), "missing data/noise/mixed/manifest.jsonl -- snr_sweep.py did not run cleanly"
with open(manifest_path) as f:
    n_mixed = sum(1 for _ in f)
assert n_mixed > 0
print(f"[PASS] {n_mixed} SNR-mixed clips ready for the noisy ASR condition")

In [ ]:
!python scripts/benchmark_asr.py --model-sizes distil-large-v3 large-v3 small --compute-types int8 float16 --csv-path outputs/asr_benchmark_gpu.csv

In [ ]:
# --- validation ---
asr_df = pd.read_csv("outputs/asr_benchmark_gpu.csv")

expected_sizes = {"distil-large-v3", "large-v3", "small"}
got_sizes = set(asr_df["model_size"].unique())
assert expected_sizes.issubset(got_sizes), f"missing model sizes: {expected_sizes - got_sizes}"
assert (asr_df["device"] == "cuda").all(), "not every row reports device=cuda"
assert not asr_df["wer"].isna().all(), "no WER values recorded"

print(f"[PASS] {len(asr_df)} real GPU rows in outputs/asr_benchmark_gpu.csv, sizes: {sorted(got_sizes)}")
print()
print(asr_df[["model_size", "compute_type", "condition", "wer", "rtf_p50", "latency_p50_ms", "latency_p95_ms", "vram_used_gb"]].to_string(index=False))

In [ ]:
!git add outputs/asr_benchmark_gpu.csv
!git commit -m "Phase 2.1: real GPU numbers (Colab T4)"
!git push origin master

In [ ]:
# --- validation: push actually landed (hard gate, per instruction not to proceed otherwise) ---
subprocess.run(["git", "fetch", "origin"], check=True)
local_head = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
remote_head = subprocess.run(["git", "rev-parse", "origin/master"], capture_output=True, text=True).stdout.strip()
assert local_head == remote_head, f"Phase 2.1 push did not land -- local {local_head[:8]} != origin/master {remote_head[:8]}. Fix before continuing to the next section."
print(f"[PASS] Phase 2.1 pushed successfully, origin/master now at {remote_head[:8]}")

## 13. Phase 2.2 — real LLM benchmark (vLLM)

`Qwen2.5-7B-Instruct-AWQ` vs `Qwen2.5-3B-Instruct-AWQ` (bigger-vs-smaller-AWQ, not the
quantized-vs-unquantized comparison this script defaulted to before) × 2 prompt lengths →
TTFT (primary metric; tok/s recorded too but not the focus), plus prefix caching on/off across
the same 6-turn conversation. Neither vLLM nor its model downloads have ever run against a real
install before this -- `scripts/benchmark_llm.py` was written against vLLM's documented
`AsyncLLMEngine` API and syntax-checked only. Report whatever actually happens, including API
mismatches -- that's real information, not a failure to hide.

**Why 4 separate cells instead of 1.** The original single-call form creates up to 4
`vllm.AsyncLLMEngine` instances sequentially in one Python process (7B prompt-lengths, 3B
prompt-lengths, 7B caching-off, 7B caching-on). This is a known vLLM failure mode on
memory-constrained cards -- each engine grabs ~90% of currently-free VRAM for its KV cache,
and that isn't reliably released back when a Python object goes out of scope (see
vllm-project/vllm issues #654 and #14376: "second model requires more memory... No available
memory for the cache blocks"). On a 16GB T4 the single-process form will likely OOM on the
2nd-4th engine. Each cell below runs one engine in its own process instead -- a full process
exit guarantees the GPU memory is actually freed before the next one starts. All 4 append to
the same `outputs/llm_benchmark_gpu.csv` / `outputs/llm_prefix_caching_gpu.csv`, so the
validation cell after them reads the full matrix exactly as before.


In [ ]:
!pip install -q vllm

In [ ]:
# --- validation ---
import vllm
print(f"[PASS] vllm {vllm.__version__} importable")

In [ ]:
!python scripts/benchmark_llm.py --model Qwen/Qwen2.5-7B-Instruct-AWQ --mode prompt-lengths

In [ ]:
# --- validation ---
import pandas as pd
df = pd.read_csv("outputs/llm_benchmark_gpu.csv")
assert "Qwen/Qwen2.5-7B-Instruct-AWQ" in set(df["model"]), "7B prompt-length rows missing"
print("[PASS] 7B prompt-length rows present")

In [ ]:
!python scripts/benchmark_llm.py --model Qwen/Qwen2.5-3B-Instruct-AWQ --mode prompt-lengths

In [ ]:
# --- validation ---
import pandas as pd
df = pd.read_csv("outputs/llm_benchmark_gpu.csv")
assert "Qwen/Qwen2.5-3B-Instruct-AWQ" in set(df["model"]), "3B prompt-length rows missing"
print("[PASS] 3B prompt-length rows present")

In [ ]:
!python scripts/benchmark_llm.py --model Qwen/Qwen2.5-7B-Instruct-AWQ --mode prefix-caching --caching off

In [ ]:
# --- validation ---
import pandas as pd
df = pd.read_csv("outputs/llm_prefix_caching_gpu.csv")
assert (df["enable_prefix_caching"] == False).any(), "caching=off rows missing"
print("[PASS] caching=off rows present")

In [ ]:
!python scripts/benchmark_llm.py --model Qwen/Qwen2.5-7B-Instruct-AWQ --mode prefix-caching --caching on

In [ ]:
# --- validation ---
import pandas as pd
df = pd.read_csv("outputs/llm_prefix_caching_gpu.csv")
assert (df["enable_prefix_caching"] == True).any(), "caching=on rows missing"
print("[PASS] caching=on rows present")

In [ ]:
# --- validation ---
llm_df = pd.read_csv("outputs/llm_benchmark_gpu.csv")
prefix_df = pd.read_csv("outputs/llm_prefix_caching_gpu.csv")

expected_models = {"Qwen/Qwen2.5-7B-Instruct-AWQ", "Qwen/Qwen2.5-3B-Instruct-AWQ"}
got_models = set(llm_df["model"].unique())
assert expected_models == got_models, f"expected {expected_models}, got {got_models}"
assert set(llm_df["prompt_length"].unique()) == {"short", "long"}
assert not llm_df["ttft_p50_ms"].isna().all(), "no TTFT values recorded"
assert set(prefix_df["enable_prefix_caching"].unique()) == {True, False}

print(f"[PASS] {len(llm_df)} prompt-length rows, {len(prefix_df)} prefix-caching rows")
print()
print("TTFT by model x prompt length:")
print(llm_df[["model", "prompt_length", "ttft_p50_ms", "ttft_p95_ms", "tok_s_p50"]].to_string(index=False))
print()
mean_ttft_by_caching = prefix_df[prefix_df["turn_index"] >= 1].groupby("enable_prefix_caching")["ttft_ms"].mean()
print("mean TTFT, turns 1-5 (turn 0 has no shared prefix to reuse), by prefix caching:")
print(mean_ttft_by_caching)

In [ ]:
!git add outputs/llm_benchmark_gpu.csv outputs/llm_prefix_caching_gpu.csv
!git commit -m "Phase 2.2: real GPU numbers (Colab T4)"
!git push origin master

In [ ]:
# --- validation: push actually landed (hard gate, per instruction not to proceed otherwise) ---
subprocess.run(["git", "fetch", "origin"], check=True)
local_head = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
remote_head = subprocess.run(["git", "rev-parse", "origin/master"], capture_output=True, text=True).stdout.strip()
assert local_head == remote_head, f"Phase 2.2 push did not land -- local {local_head[:8]} != origin/master {remote_head[:8]}. Fix before continuing to the next section."
print(f"[PASS] Phase 2.2 pushed successfully, origin/master now at {remote_head[:8]}")

## 14. Phase 2.3 — real TTS benchmark (Kokoro + Piper, GPU)

Kokoro/Piper already have real, measured CPU numbers from earlier in this project -- this adds
real T4 numbers alongside them. Piper takes an explicit `--use-cuda` flag; Kokoro auto-detects a
GPU via the `onnxruntime-gpu` package (no code change needed, just install it). 3 sentences ×
3 repeats, time-to-first-chunk only.

In [ ]:
!pip uninstall -y onnxruntime -q
!pip install -q onnxruntime-gpu

In [ ]:
# --- validation ---
import onnxruntime
providers = onnxruntime.get_available_providers()
print(f"available onnxruntime providers: {providers}")
assert "CUDAExecutionProvider" in providers, "onnxruntime-gpu install didn't expose CUDAExecutionProvider"
print("[PASS] CUDAExecutionProvider available")

In [ ]:
!python scripts/prepare_tts_models.py

In [ ]:
# --- validation ---
kokoro_model = Path("data/tts/models/kokoro/kokoro-v1.0.int8.onnx")
piper_model = Path("data/tts/models/piper/en_US-lessac-medium.onnx")
assert kokoro_model.exists() and piper_model.exists(), "TTS model files missing"
print("[PASS] Kokoro + Piper model files present")

In [ ]:
!python scripts/benchmark_tts.py --engines kokoro piper --use-cuda

In [ ]:
# --- validation ---
tts_gpu_df = pd.read_csv("outputs/tts_benchmark_gpu.csv")

assert set(tts_gpu_df["engine"].unique()) == {"kokoro", "piper"}, f"expected both engines, got {tts_gpu_df['engine'].unique()}"
assert (tts_gpu_df["device"] == "cuda").all(), "not every row reports device=cuda"

print("[PASS] real GPU TTS benchmark complete, all rows report device=cuda")
print()
print(tts_gpu_df[["engine", "sentence_index", "ttfc_p50_ms", "ttfc_p95_ms"]].to_string(index=False))

try:
    cpu_df = pd.read_csv("outputs/tts_benchmark.csv")
    print("\nmean TTFC p50 by engine, CPU vs GPU (ms):")
    print(pd.DataFrame({"cpu": cpu_df.groupby("engine")["ttfc_p50_ms"].mean(), "gpu": tts_gpu_df.groupby("engine")["ttfc_p50_ms"].mean()}))
except FileNotFoundError:
    print("(no outputs/tts_benchmark.csv from an earlier CPU run in this clone to compare against)")

In [ ]:
!git add outputs/tts_benchmark_gpu.csv
!git commit -m "Phase 2.3: real GPU numbers (Colab T4)"
!git push origin master

In [ ]:
# --- validation: push actually landed (hard gate, per instruction not to proceed otherwise) ---
subprocess.run(["git", "fetch", "origin"], check=True)
local_head = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
remote_head = subprocess.run(["git", "rev-parse", "origin/master"], capture_output=True, text=True).stdout.strip()
assert local_head == remote_head, f"Phase 2.3 push did not land -- local {local_head[:8]} != origin/master {remote_head[:8]}. Fix before continuing to the next section."
print(f"[PASS] Phase 2.3 pushed successfully, origin/master now at {remote_head[:8]}")

## 15. Phase 0.4 — hardware baseline, T4 side

`scripts/hardware_baseline.py` already has a real CPU row (zenbook-cpu, run locally). This
adds the real Colab T4 row to the same CSV -- VRAM, a synthetic-workload TTFT proxy, and
throughput, so `outputs/hardware_baseline.csv` has both machines this project actually runs
on side by side. No model weights involved, runs in seconds.


In [ ]:
!python scripts/hardware_baseline.py --label colab-t4

In [ ]:
# --- validation ---
import pandas as pd
hb_df = pd.read_csv("outputs/hardware_baseline.csv")
row = hb_df[hb_df["label"] == "colab-t4"]
assert len(row) >= 1, "no colab-t4 row written"
assert (row["device_type"] == "cuda").all(), "colab-t4 row didn't report device_type=cuda"
print("[PASS] colab-t4 row present, device_type=cuda")
print(row.to_string(index=False))

In [ ]:
!git add outputs/hardware_baseline.csv
!git commit -m "Phase 0.4: real T4 hardware baseline (Colab)"
!git push origin master

In [ ]:
# --- validation: push actually landed (hard gate, per instruction not to proceed otherwise) ---
subprocess.run(["git", "fetch", "origin"], check=True)
local_head = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
remote_head = subprocess.run(["git", "rev-parse", "origin/master"], capture_output=True, text=True).stdout.strip()
assert local_head == remote_head, f"Phase 0.4 push did not land -- local {local_head[:8]} != origin/master {remote_head[:8]}. Fix before continuing."
print(f"[PASS] Phase 0.4 pushed successfully, origin/master now at {remote_head[:8]}")

## Cleanup

Removes the token from git config now that all four sections have pushed -- no reason to leave
a live credential sitting in `.git/config` for the rest of the session.


In [ ]:
subprocess.run(["git", "remote", "set-url", "origin", "https://github.com/varshitthhh/voice-ai-pipeline.git"], check=True)
print("remote URL reset, token removed from git config")

## Summary

- **Phase 2.1**: full production ASR matrix (`distil-large-v3`/`large-v3`/`small` ×
  `int8`/`float16`, clean+noisy) with real WER/RTF/VRAM/p50-p95, pushed to
  `outputs/asr_benchmark_gpu.csv`.
- **Phase 2.2**: `Qwen2.5-7B-Instruct-AWQ` vs `Qwen2.5-3B-Instruct-AWQ` on real vLLM, TTFT across
  2 prompt lengths plus prefix-caching on/off across a 6-turn conversation, pushed to
  `outputs/llm_benchmark_gpu.csv` / `outputs/llm_prefix_caching_gpu.csv` -- run as 4 separate
  single-engine processes to avoid vLLM's known multi-engine GPU-memory-not-released issue on a
  16GB T4 (vllm-project/vllm #654, #14376).
- **Phase 2.3**: Kokoro + Piper time-to-first-chunk on real T4 GPU, pushed to
  `outputs/tts_benchmark_gpu.csv`.
- **Phase 0.4**: real T4 row (VRAM, TTFT proxy, throughput) alongside the existing real CPU row,
  pushed to `outputs/hardware_baseline.csv`.
- Each section pushed and was verified (local HEAD == origin/master) before the next one started.
- **Still not real**: the ASR noisy condition uses SLURP+MUSAN, not CANDOR audio -- re-run once
  Phase 1.3 is redone against real CANDOR audio, whenever that access comes through.
